# 04 Phase Retrieval

            Loads the HDF5 data dictionary, runs the unified recipe-driven phase
            retrieval, plots the CDI results, and writes results and recipe back
            into the same HDF5 file.

In [1]:
import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector


def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())


BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf
wf = reload(wf)  # Refresh helpers when rerunning in an existing kernel.

try:
    import cupy as cp
    import cupyx as cpx
    import CCI_core_cupy as cci
    import Phase_Retrieval as PhR

    GPU = True
    print("GPU available")
except Exception:
    import CCI_core as cci

    PhR = None
    GPU = False
    print("GPU unavailable")

%matplotlib widget
try:
    %load_ext jupyter_black
except Exception:
    pass

  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/riccardo/anaconda3/envs/myenv/lib/python3.14/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/riccardo/anaconda3/envs/myenv/lib/python3.14/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/Users/riccardo/anaconda3/envs/myenv/lib/python3.14/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/Users/riccardo/anaconda3/envs/myenv/lib/python3.14/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/Users/riccardo/anaconda3/envs/myenv/lib/python3.14/asyncio/base_events.py", line 677, in run_forever
    self._run_once()
  File "/Users/riccardo/anaconda3/envs/myenv/lib/python3.14/asyncio/base_events.py", line 2057, in _run_once
    handle._run()
  File "/Users/riccardo/anaconda

Base folder: /Users/riccardo/Github/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS
GPU unavailable


In [2]:
import phase_retrieval_core_unified as pr

In [3]:
def selected_mode(recon, mode=0):
    modes = wf.as_modes(recon)
    mode = min(int(mode), modes.shape[0] - 1)
    return modes[mode]

## Load data

In [5]:
BASEFOLDER = find_basefolder()
USER = "rb"
im_id = 95    # Positive-helicity image ID produced by 01_FTH.ipynb.
topo_id = 96  # Reference/negative-helicity image ID in the same HDF5 file.
# True passes the centered mask_pixel saved by 01_FTH to phase retrieval.
# Masked detector pixels are excluded from the image constraint.
USE_MASK_PIXEL = True
folder_general = helper.create_folder(join(BASEFOLDER, "processed"))
folder_logs = helper.create_folder(join(folder_general, "Logs"))
DATA_H5 = join(folder_logs, f"data_recon_ImId_{im_id:04d}_{USER}.hdf5")

data = wf.load_data_dict(DATA_H5)
experimental_setup = data["experimental_setup"]
positive_label = data["positive_label"]
reference_label = data["reference_label"]
loaded_im_id = int(data["holo"][positive_label]["id"])
loaded_topo_id = int(data["holo"][reference_label]["id"])
if loaded_im_id != im_id:
    raise ValueError(f"Requested im_id={im_id}, but the HDF5 contains im_id={loaded_im_id}.")
if loaded_topo_id != topo_id:
    raise ValueError(f"Requested topo_id={topo_id}, but the HDF5 contains topo_id={loaded_topo_id}.")
labels = wf.get_hologram_labels(data)
print("Phase retrieval labels:", labels)
pol1, pol2 = "+", "-"
phase_retrieval_labels = [label for label in (pol1, pol2) if label in labels]
phase_retrieval_labels += [
    label for label in labels if label not in phase_retrieval_labels
]
if pol1 not in labels or pol2 not in labels:
    raise ValueError(
        f'The FTH_CDI_01 recipe expects "{pol1}" and "{pol2}" in data["holo"]. '
        f"Available labels are {labels}."
    )

if "supportmask" not in data:
    raise ValueError("Run 03_define_supportmask.ipynb before phase retrieval.")

supportmask = np.asarray(data["supportmask"])
# data['mask_pixel'] was already centered in 01_FTH; do not center it twice.
stored_mask_pixel = np.asarray(
    data.get("mask_pixel", np.zeros_like(supportmask)), dtype=np.uint8
)
if stored_mask_pixel.shape != supportmask.shape:
    raise ValueError(
        f"mask_pixel shape {stored_mask_pixel.shape} != supportmask shape {supportmask.shape}"
    )
mask_pixel = (
    stored_mask_pixel
    if USE_MASK_PIXEL
    else np.zeros_like(stored_mask_pixel, dtype=np.uint8)
)
print(f"mask_pixel filtering: {'enabled' if USE_MASK_PIXEL else 'disabled'}")
focus = dict(data.get("focus", {}))
focus_cdi = dict(data.get("focus_cdi", {}))
if "roi" in focus_cdi:
    roi_cdi = focus_cdi["roi"]
    roi_cdi_source = 'focus_cdi["roi"]'
elif "roi" in focus:
    roi_cdi = focus["roi"]
    roi_cdi_source = 'focus["roi"]'
else:
    roi_cdi = [0, supportmask.shape[0], 0, supportmask.shape[1]]
    roi_cdi_source = "full supportmask"
roi_cdi = np.asarray(roi_cdi, dtype=int)
print(f"roi_cdi from {roi_cdi_source}:", roi_cdi)
roi_cdi_s = wf.roi_to_slices(roi_cdi)
holograms = {
    label: np.asarray(data["holo"][label]["image_c"])
    for label in phase_retrieval_labels
}

Phase retrieval labels: ['+', '-']
mask_pixel filtering: enabled
roi_cdi from full supportmask: [   0 2048    0 2048]


## Recipe

In [6]:
# Exact formulation from FTH_CDI_01.ipynb.
offset_vmin = 0.1
Startimage = None
Startgamma = None

phase_retrieval_holograms = {
    label: holograms[label] for label in phase_retrieval_labels
}
primary_label = phase_retrieval_labels[0]
secondary_labels = phase_retrieval_labels[1:]
full_labels = [primary_label, primary_label, *secondary_labels]
partial_labels = [primary_label, primary_label, *secondary_labels]
recipe_labels = full_labels + partial_labels

times = 1
phase_retrieval_recipe = {
    "algorithm_list": ["HAPRE", "ER", "ER"] * times,
    "number_iterations": [100, 50, 50] * times,
    "helicity": ["+", "+", "-"] * times,
    "beta_zero": 0.5,
    "beta_mode": ["arctan", "const", "const"] * times,
    "alpha_zero": 0.0,
    "alpha_mode": "const",
    "RL_its": [0, 0, 0, 50, 50, 50][: 3 * times],
    "RL_freqs": [1e9, 1e9, 1e9, 20, 20, 20][: 3 * times],
    "TV_freqs": 1e9,
    "plot_every": 50,
    "average_img": 10,
    "Fourier_last": True,
    "Startimage": [None, "+", "+", "+", "+", "+"][: 3 * times],
    "Startgamma": [None, None, None, None, "+", "+"][: 3 * times],
    "hologram_intensity_cutoff_vmin": 0.01,
    "output": [False, True, True, False, True, True][: 3 * times],
    "modes": [1, 2],
    "normalize_startimage_between_holograms": True,
    "return_format": "auto",
    "crop": 100,
}
phase_retrieval_recipe

{'algorithm_list': ['HAPRE', 'ER', 'ER'],
 'number_iterations': [100, 50, 50],
 'helicity': ['+', '+', '-'],
 'beta_zero': 0.5,
 'beta_mode': ['arctan', 'const', 'const'],
 'alpha_zero': 0.0,
 'alpha_mode': 'const',
 'RL_its': [0, 0, 0],
 'RL_freqs': [1000000000.0, 1000000000.0, 1000000000.0],
 'TV_freqs': 1000000000.0,
 'plot_every': 50,
 'average_img': 10,
 'Fourier_last': True,
 'Startimage': [None, '+', '+'],
 'Startgamma': [None, None, None],
 'hologram_intensity_cutoff_vmin': 0.01,
 'output': [False, True, True],
 'modes': [1, 2],
 'normalize_startimage_between_holograms': True,
 'return_format': 'auto',
 'crop': 100}

## Run phase retrieval

In [7]:
phase_retrieval_result = pr.phase_retrieval_algorithm(
    phase_retrieval_holograms,
    mask_pixel,
    supportmask,
    phase_retrieval_recipe,
)

retrieved_holograms = phase_retrieval_result["full_coherence"]
retrieved_holograms_pc = phase_retrieval_result["partial_coherence"]
retrieved_holograms_gradient = phase_retrieval_result["gradient_descent"]
bsmasks = phase_retrieval_result["bsmasks"]
gammas = phase_retrieval_result["gamma"]
errors = phase_retrieval_result["error"]
print("Phase retrieval done.")

for label in phase_retrieval_labels:
    data["holo"][label]["retrieved_full"] = retrieved_holograms.get(label)
    data["holo"][label]["retrieved_pc"] = retrieved_holograms_pc.get(label)
    data["holo"][label]["retrieved_gradient"] = retrieved_holograms_gradient.get(label)
    data["holo"][label]["bsmask"] = bsmasks.get(label)
    data["holo"][label]["gamma"] = gammas.get(label)

error_summary = {
    "steps": [
        {
            "step": step["step"],
            "helicity": step["helicity"],
            "mode": step["mode"],
            "Nit": step["Nit"],
            "RL_it": step["RL_it"],
            "RL_freq": step["RL_freq"],
            "coherence": step["coherence"],
            "output": step["output"],
            "error": np.asarray(step["error"]),
            "support_error": np.asarray(step["support_error"]),
        }
        for step in errors["steps"]
    ],
}

ValueError: Unknown phase-retrieval recipe key(s): ['crop', 'modes']

## CDI reconstruction and focus

In [ ]:
pol1 = "+"
pol2 = "-"
retrieved_type = (
    "retrieved_full"  # options: "retrieved_full", "retrieved_pc", "retrieved_gradient"
)

for pol in (pol1, pol2):
    available = [
        key
        for key in ("retrieved_full", "retrieved_pc", "retrieved_gradient")
        if data["holo"][pol].get(key) is not None
    ]
    print(pol, "available:", available)

holo1 = data["holo"][pol1][retrieved_type]
holo2 = data["holo"][pol2][retrieved_type]
if holo1 is None or holo2 is None:
    raise ValueError(f"{retrieved_type} is not available for {pol1} and/or {pol2}.")
holo1 = np.asarray(holo1)
holo2 = np.asarray(holo2)

phase_cdi = focus_cdi.get("phase", 0)
prop_dist_cdi = focus_cdi.get("prop_dist", 0)
dx = focus_cdi.get("dx", 0)
dy = focus_cdi.get("dy", 0)
focus_mode_cdi = int(focus_cdi.get("mode", 0))
use_bs = False
bs_diam_cdi = 25

retrieved_shape = wf.spatial_shape(holo1)
if use_bs:
    mask_bs_cdi = 1 - cci.circle_mask(
        retrieved_shape, np.array(retrieved_shape) / 2, bs_diam_cdi, sigma=4
    )
else:
    mask_bs_cdi = np.ones(retrieved_shape)

crop = int(phase_retrieval_recipe["crop"])
crop_shape = np.array(supportmask.shape) - 2 * crop
if tuple(crop_shape) != retrieved_shape:
    raise ValueError(
        f"Expected retrieved shape {tuple(crop_shape)}, got {retrieved_shape}."
    )

scale = crop_shape / np.array(supportmask.shape)
roi_crop = np.rint(np.array(roi_cdi) * [scale[0], scale[0], scale[1], scale[1]]).astype(
    int
)
roi_crop[[0, 1]] = np.clip(roi_crop[[0, 1]], 0, crop_shape[0])
roi_crop[[2, 3]] = np.clip(roi_crop[[2, 3]], 0, crop_shape[1])
if roi_crop[1] <= roi_crop[0] or roi_crop[3] <= roi_crop[2]:
    raise ValueError(f"Invalid scaled CDI ROI: {roi_crop} from roi_cdi {roi_cdi}.")
roi_crop_s = wf.roi_to_slices(roi_crop)
roi_crop_shape = (roi_crop[1] - roi_crop[0], roi_crop[3] - roi_crop[2])
print("roi_cdi:", roi_cdi)
print("roi_crop:", roi_crop, "shape:", roi_crop_shape, "of", tuple(crop_shape))

p_cdi_all = wf.reconstruct_cdi_modes(
    holo1,
    mask_bs_cdi,
    fth,
    experimental_setup,
    prop_dist=prop_dist_cdi,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
)
n_cdi_all = wf.reconstruct_cdi_modes(
    holo2,
    mask_bs_cdi,
    fth,
    experimental_setup,
    prop_dist=prop_dist_cdi,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
)
p_cdi = selected_mode(p_cdi_all, focus_mode_cdi)
n_cdi = selected_mode(n_cdi_all, focus_mode_cdi)
print("phase_cdi:", phase_cdi)
print("prop_dist_cdi:", prop_dist_cdi)
print("focus_mode_cdi:", focus_mode_cdi)

In [ ]:
# Optional fine tuning. Use the sliders, then execute the following cell to store values.
mode = "-"
supportmask_eff = wf.resize_binary_to_shape(supportmask, wf.spatial_shape(p_cdi))
focus_sliders = rec.focusCDI(
    holo1 * mask_bs_cdi,
    holo2 * mask_bs_cdi,
    roi_crop_s,
    mask=supportmask_eff,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
    prop_dist=prop_dist_cdi,
    experimental_setup=experimental_setup,
    operation=mode,
    max_prop_dist=3,
    scale=(2, 98),
)
slider_prop, slider_phase, slider_dx, slider_dy = focus_sliders[:4]
slider_mode = focus_sliders[4] if len(focus_sliders) > 4 else None

In [ ]:
phase_cdi = slider_phase.value
prop_dist_cdi = slider_prop.value
dx = slider_dx.value
dy = slider_dy.value
focus_mode_cdi = int(slider_mode.value) if slider_mode is not None else 0

p_cdi_all = wf.reconstruct_cdi_modes(
    holo1,
    mask_bs_cdi,
    fth,
    experimental_setup,
    prop_dist=prop_dist_cdi,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
)
n_cdi_all = wf.reconstruct_cdi_modes(
    holo2,
    mask_bs_cdi,
    fth,
    experimental_setup,
    prop_dist=prop_dist_cdi,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
)
p_cdi = selected_mode(p_cdi_all, focus_mode_cdi)
n_cdi = selected_mode(n_cdi_all, focus_mode_cdi)
print("Updated phase_cdi:", phase_cdi)
print("Updated prop_dist_cdi:", prop_dist_cdi)
print("Updated focus_mode_cdi:", focus_mode_cdi)

## Plot and save

In [ ]:
recon_cdi_full = np.log(p_cdi) - np.log(n_cdi)
cdi_shape = wf.spatial_shape(recon_cdi_full)
supportmask_eff = wf.resize_binary_to_shape(supportmask, cdi_shape)
recon_cdi_roi = wf.spatial_roi(recon_cdi_full, roi_crop_s)
supportmask_roi = supportmask_eff[roi_crop_s]
recon_cdi = wf.apply_spatial_mask(recon_cdi_roi, supportmask_roi)
print("Plotting CDI ROI:", roi_crop, "shape:", recon_cdi.shape)

fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(recon_cdi)
vmin, vmax = wf.finite_percentile_limits(tmp)
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(f"{pol1} - {pol2} {retrieved_type}, mode {focus_mode_cdi}")
ax.set_axis_off()

png_name = join(folder_general, f"PhR_recon_ImId_{int(im_id):04d}_{USER}.png")
plt.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.show()
print("Saved figure:", png_name)

In [ ]:
png_name = join(folder_general, f"PhR_recon_ImId_{int(im_id):04d}_{USER}.png")

# Always write the display PNG in the same final cell as the HDF5 result.
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(recon_cdi)
vmin, vmax = wf.finite_percentile_limits(tmp)
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(f"{pol1} - {pol2} {retrieved_type}, mode {focus_mode_cdi}")
ax.set_axis_off()
fig.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.close(fig)

focus_cdi = {
    "prop_dist": prop_dist_cdi,
    "phase": phase_cdi,
    "dx": dx,
    "dy": dy,
    "roi": roi_cdi,
    "roi_crop": roi_crop,
    "mode": focus_mode_cdi,
    "operation": mode,
    "retrieved_type": retrieved_type,
    "pol1": pol1,
    "pol2": pol2,
}

for key in [
    "dark_id_im",
    "dark_id_topo",
    "im_id",
    "topo_id",
    "fth_hologram",
    "fth_hologram_unmasked",
    "fth_png_title",
    "fth_recon",
    "fth_recon_unmasked",
    "mask_pixel_smooth",
    "mask_multiplier",
    "sum_c",
    "diff_c",
    "mask_pixel_c",
    "mask_pixel_c_png",
    "prop_dist",
    "phase",
    "dx",
    "dy",
    "focus_operation",
    "roi",
    "recon_cdi",
    "recon_topo_cdi",
    "phase_retrieval_png",
    "roi_cdi",
    "retrieved_type",
    "phase_cdi",
    "prop_dist_cdi",
    "dx_cdi",
    "dy_cdi",
    "focus_mode_cdi",
    "roi_crop",
    "mask_bs_cdi",
]:
    data.pop(key, None)
data["phase_retrieval_recipe"] = phase_retrieval_recipe
data["phase_retrieval_errors"] = error_summary
data["focus_cdi"] = focus_cdi
data["recon_cdi"] = recon_cdi
data["phase_retrieval_png"] = png_name

wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Saved HDF5:", DATA_H5)
print("Saved PNG:", png_name)

In [ ]:
# Workflow summary
_summary_data = data if "data" in globals() and isinstance(data, dict) else {}
_summary_h5 = globals().get("DATA_H5", _summary_data.get("data_file", "n/a"))
_summary_holo = _summary_data.get("holo", {})
_summary_pos = _summary_data.get(
    "positive_label", globals().get("positive_label", None)
)
_summary_ref = _summary_data.get(
    "reference_label", globals().get("reference_label", None)
)
_summary_im = _summary_holo.get(_summary_pos, {}).get(
    "id", globals().get("im_id", "n/a")
)
_summary_topo = _summary_holo.get(_summary_ref, {}).get(
    "id", globals().get("topo_id", "n/a")
)
print("im_id:", _summary_im)
print("topo_id:", _summary_topo)
print("HDF5:", _summary_h5)